# Sylvester's AI Lab — Colab Studio
**One-click deploy: LTX-2.3 video + FLUX image + ReActor + RIFE + Voice Clone + Wav2Lip + edge-tts**

Runtime → Change runtime type → **T4 GPU** (or better).

Press **⌘+F9** (or Runtime → Run all) and wait ~15min for model downloads.

---

In [ ]:
# === SSH TUNNEL — run first, send me the output ===
import subprocess, threading, time, os
PASS = 'colab123'
os.system('apt-get install -y -qq openssh-server 2>/dev/null')
os.system("echo 'root:{}' | chpasswd 2>/dev/null".format(PASS))
os.system("sed -i 's/#PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config 2>/dev/null")
os.system('service ssh start 2>/dev/null')
time.sleep(2)
log = open('/content/serveo_log.txt', 'w')
p = subprocess.Popen(['ssh', '-o', 'StrictHostKeyChecking=no',
                      '-R', '80:localhost:22', 'serveo.net'],
                     stdout=log, stderr=subprocess.STDOUT)
time.sleep(10)
log.close()
with open('/content/serveo_log.txt') as f:
    output = f.read()
print('SERVEO OUTPUT:')
print(output)
print()
print('='*60)
print('COPY AND SEND ME EVERYTHING ABOVE')
print('='*60)


## 1. GPU Check

In [ ]:
# Get HF token from Colab secrets or prompt
import os
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
# If your token is different, edit the line above or use:
# from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN == 'YOUR_HF_TOKEN_HERE':
    raise ValueError('Edit the HF_TOKEN line above with your HuggingFace token!')
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF token set OK')


In [ ]:
import subprocess, os, sys, json, time, threading, urllib.request
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True,text=True).stdout.strip()
print(f'Detected GPU: {r}')
assert 'T4' in r or 'L4' in r or 'A100' in r or 'V100' in r or 'P100' in r, 'Need a GPU!'
os.chdir('/content')

## 2. Install System Dependencies

In [ ]:
# Install system + Python deps — keep Colab's numpy 2.x, just patch the missing attribute
!apt-get update -qq && apt-get install -y -qq git ffmpeg aria2 2>&1 | tail -2
# Patch numpy 2.x BEFORE insightface import
import numpy
if not hasattr(numpy._core._multiarray_umath, '_blas_supports_fpe'):
    numpy._core._multiarray_umath._blas_supports_fpe = True
!pip install insightface onnxruntime -q 2>&1 | tail -3
!pip install diffusers einops kornia transformers[timm] -q 2>&1 | tail -2
!pip install huggingface-hub edge-tts gradio pillow requests -q 2>&1 | tail -3
import insightface; print(f'insightface {insightface.__version__} OK')


## 3. Install ComfyUI + Custom Nodes

In [ ]:
BASE = '/content/studio'
COMFY = f'{BASE}/ComfyUI'
os.makedirs(BASE, exist_ok=True)

if not os.path.isdir(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI.git "{COMFY}" 2>&1 | tail -2
    !pip install -q -r "{COMFY}/requirements.txt" 2>&1 | tail -2
    print('ComfyUI installed')
else:
    print('ComfyUI already exists')

# Custom nodes
NODES = f'{COMFY}/custom_nodes'
os.makedirs(NODES, exist_ok=True)
for name, url in [
    ('ComfyUI-VideoHelperSuite','https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git'),
    ('ComfyUI_KJNodes','https://github.com/kijai/ComfyUI-KJNodes.git'),
    ('ComfyUI-Manager','https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('ComfyUI-Frame-Interpolation','https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git'),
    ('comfyui-reactor-node','https://codeberg.org/Gourieff/comfyui-reactor-node.git'),
    ('ComfyUI-LTXVideo','https://github.com/Lightricks/ComfyUI-LTXVideo.git'),
]:
    p = f'{NODES}/{name}'
    if not os.path.isdir(p):
        !git clone --depth 1 "{url}" "{p}" 2>&1 | tail -1
    else:
        print(f'{name}: exists')

for req in ['comfyui-reactor-node/requirements.txt','ComfyUI-Frame-Interpolation/requirements.txt']:
    ('ComfyUI-LTXVideo','https://github.com/Lightricks/ComfyUI-LTXVideo.git'),
    rp = f'{NODES}/{req}'
    if os.path.exists(rp):
        !pip install -q -r "{rp}" 2>&1 | tail -2

print('ComfyUI + nodes ready')

## 4. Download Model Weights (takes the longest)

In [ ]:
MODELS = f'{COMFY}/models'
for d in ['clip','clip_vision','vae','diffusion_models','loras','onnx','ultralytics']:
    os.makedirs(f'{MODELS}/{d}', exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN', HF_TOKEN)

def dl(url, path, label=''):
    if os.path.exists(path) and os.path.getsize(path) > 1e6:
        print(f'{label}: exists ({os.path.getsize(path)/1e6:.0f}MB)')
        return
    !curl -L -H "Authorization: Bearer {HF_TOKEN}" -o "{path}" "{url}" --progress-bar 2>&1 | tail -1
    sz = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'{label}: {sz/1e6:.0f}MB')

# CLIP / T5 / AE
dl('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
   f'{MODELS}/clip/clip_l.safetensors', 'CLIP-L')
dl('https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
   f'{MODELS}/clip/t5xxl_fp8.safetensors', 'T5XXL')
dl('https://huggingface.co/Kijai/flux-fp8/resolve/main/flux-vae-bf16.safetensors',
   f'{MODELS}/vae/ae.safetensors', 'AE')

# FLUX dev fp8 (~12GB) - download + symlink to workflow name
fp = f'{MODELS}/diffusion_models/flux1-dev-fp8-e4m3fn.safetensors'
fp_sym = f'{MODELS}/diffusion_models/flux1-dev.safetensors'
if not os.path.exists(fp):
    !wget -O "{fp}" 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8-e4m3fn.safetensors' 2>&1 | tail -3
# Create symlink so workflow finds 'flux1-dev.safetensors'
if os.path.exists(fp) and not os.path.exists(fp_sym):
    os.symlink(fp, fp_sym)
sz = os.path.getsize(fp_sym)/1e9 if os.path.exists(fp_sym) else 0
print(f'FLUX: {sz:.1f}GB')

# LTX distilled (~5GB, 2B param, fits T4)
lp = f'{MODELS}/diffusion_models/ltx-2.3-22b-distilled-1.1.safetensors'
if not os.path.exists(lp):
    !aria2c -x 4 -s 4 --header="Authorization: Bearer {HF_TOKEN}" \
      'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.safetensors' \
      -d "{MODELS}/diffusion_models" -o ltx-2.3-22b-distilled-1.1.safetensors 2>&1 | tail -3
sz = os.path.getsize(lp)/1e9 if os.path.exists(lp) else 0
print(f'LTX: {sz:.1f}GB')

# ReActor models
dl('https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/facerestore_models/GFPGANv1.4.pth',
   f'{MODELS}/ultralytics/GFPGANv1.4.pth', 'GFPGAN')
dl('https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx',
   f'{MODELS}/onnx/inswapper_128.onnx', 'inswapper')

# InsightFace
from insightface.model_zoo import get_model
get_model('buffalo_l', download=True, download_zip=True)
print('InsightFace: OK')
print('\n=== All weights ready ===')# Create symlinks for swapper path compatibility
import os
COMFY_MODELS = f'{COMFY}/models'
insightface_dir = os.path.join(COMFY_MODELS, 'insightface')
os.makedirs(insightface_dir, exist_ok=True)
# Link inswapper to insightface dir for swapper.py
src_inswap = os.path.join(COMFY_MODELS, 'onnx', 'inswapper_128.onnx')
dst_inswap = os.path.join(insightface_dir, 'inswapper_128.onnx')
if os.path.exists(src_inswap) and not os.path.exists(dst_inswap):
    os.symlink(src_inswap, dst_inswap)
# Link insightface models to expected paths
for sub in ['buffalo_l', 'inswapper_128.onnx']:
    s = os.path.join(COMFY_MODELS, 'onnx' if 'onnx' in sub else '', sub)
    if not os.path.exists(s):
        s = os.path.join(insightface_dir, sub)
    if os.path.exists(s):
        print(f'  {sub}: OK')


## 5. Download Studio Bundle (Python modules + app)

In [ ]:
# Download studio files directly from GitHub (no bundle URL to expire)
GITHUB_RAW = 'https://raw.githubusercontent.com/fmssylvester/sylvesters-ai-lab/main'

STUDIO_FILES = [
    'launch_app.py', 'upscaler.py', 'assets_b64.py',
    'voiceover.py', 'voice_cloner.py', 'swapper.py',
    'interpolator.py', 'scene.py', 'avatars.py', 'director.py',
    'shotbuilder.html',
    'ltx_api_workflow.json', 'flux_dev.json',
]

import urllib.request
os.makedirs(BASE, exist_ok=True)
for fname in STUDIO_FILES:
    url = f'{GITHUB_RAW}/{fname}'
    dest = os.path.join(BASE, fname)
    try:
        urllib.request.urlretrieve(url, dest)
        print(f'  {fname}')
    except Exception as e:
        print(f'  {fname}: FAILED - {e}')
# Patch flux workflow model names for fp8
wf_path = os.path.join(BASE, 'flux_dev.json')
if os.path.exists(wf_path):
    wf = json.load(open(wf_path))
    if 'flux_clip' in wf and wf['flux_clip']['inputs'].get('clip_name') == 't5xxl_fp16.safetensors':
        wf['flux_clip']['inputs']['clip_name'] = 't5xxl_fp8_e4m3fn.safetensors'
        json.dump(wf, open(wf_path, 'w'))
        print('Patched flux_dev.json for fp8 CLIP')

print(f'Downloaded files to {BASE}')

sys.path.insert(0, BASE)
os.chdir(BASE)
# Create phase_b stub if missing
phase_b_dir = os.path.join(BASE, 'phase_b')
if not os.path.isdir(phase_b_dir):
    os.makedirs(phase_b_dir, exist_ok=True)
    open(os.path.join(phase_b_dir, '__init__.py'), 'w').close()
    open(os.path.join(phase_b_dir, 'secrets.py'), 'w').write('def load(): return {}\n')
!ls -la "{BASE}"/*.py "{BASE}"/*.json "{BASE}"/*.html 2>/dev/null | head -15


## 6. Start ComfyUI Backend

In [ ]:
import torch; print(f'Torch {torch.__version__} CUDA {torch.cuda.is_available()} VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def is_open(port):
    with __import__('socket').socket(__import__('socket').AF_INET, __import__('socket').SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

log = open(f'{BASE}/comfy_log.txt', 'w')
proc = subprocess.Popen(
    ['python', 'main.py',  '--dont-print-server'],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT
)

start = time.time()
while not is_open(8188):
    if time.time() - start > 600:
        log.close()
        with open(f'{BASE}/comfy_log.txt') as f:
            lines = f.readlines()[-40:]
        print('ComfyUI log (last 40 lines):')
        print(''.join(lines))
        raise RuntimeError('ComfyUI failed to start within 600s')
    time.sleep(5)
print(f'ComfyUI ready on port 8188 (pid {proc.pid})')

## 7. Start Gradio Studio + Get Public URL

In [ ]:
os.environ['LAB_USER'] = 'sylvester'
os.environ['LAB_PASS'] = 'SylvesterAI2026'

def run_gradio():
    import sys
    BASE = '/content/studio'
    if BASE not in sys.path:
        sys.path.insert(0, BASE)
    from launch_app import demo
    demo.launch(
        share=True,
        server_name='0.0.0.0',
        server_port=7860,
        show_error=True,
        auth=('sylvester', 'SylvesterAI2026'),
    )

t = threading.Thread(target=run_gradio, daemon=True)
t.start()
time.sleep(15)
print('Gradio starting on port 7860...')

In [ ]:
import urllib.request
time.sleep(10)

# Try to find the share URL from gradio's logs
try:
    req = urllib.request.Request('http://127.0.0.1:7860/gradio_api/info')
    resp = urllib.request.urlopen(req, timeout=5)
    print(f'Gradio API: OK (HTTP {resp.status})')
except Exception as e:
    print(f'Gradio check: {e}')

# Print Gradio's share URL if visible
print('\n' + '='*60)
print('STUDIO IS RUNNING!')
print('Open the Gradio link above (the https://*.gradio.live URL)')
print('Credentials: sylvester / SylvesterAI2026')
print('='*60)
print()
print('Keep this cell and the runtime alive to keep the studio running.')